# 04 — Risk Scoring
Converts the champion model's default probability into a 0-100 risk score
and a Low/Medium/High category for every scoreable customer, and writes
`models/scored_customers.csv` -- the table the BI dashboard's high-risk
list and donut chart read from. Logic lives in `src/risk_score.py`.

In [ ]:
import sys
sys.path.append('../src')
import pandas as pd
import matplotlib.pyplot as plt

DB = '../data/credit_risk.db'
MODELS_DIR = '../models'

## 1. Run the scoring pipeline

In [ ]:
%run ../src/risk_score.py --db ../data/credit_risk.db --models-dir ../models --out ../models/scored_customers.csv

## 2. Risk category distribution

In [ ]:
scored = pd.read_csv(f'{MODELS_DIR}/scored_customers.csv')
counts = scored['risk_category'].value_counts()
print(counts)

colors = {'Low Risk': '#2ca02c', 'Medium Risk': '#ff7f0e', 'High Risk': '#d62728'}
counts.plot(
    kind='pie', autopct='%1.1f%%', figsize=(5, 5),
    colors=[colors[c] for c in counts.index],
)
plt.ylabel('')
plt.title('Customer risk distribution')
plt.show()

## 3. Risk score distribution

In [ ]:
scored['risk_score'].plot(kind='hist', bins=25, figsize=(7, 4), color='steelblue')
plt.axvline(33, color='green', linestyle='--', label='Low/Medium boundary')
plt.axvline(66, color='red', linestyle='--', label='Medium/High boundary')
plt.title('Risk score distribution across the book')
plt.xlabel('Risk score (0-100)')
plt.legend()
plt.tight_layout()
plt.show()

## 4. High-risk watchlist (top 20)

In [ ]:
scored[scored['risk_category'] == 'High Risk'].head(20)[
    ['customer_id', 'city', 'state', 'default_probability', 'risk_score']
]

## Notes
- Thresholds (0-33 Low, 34-66 Medium, 67-100 High) follow the PRD default and can be re-tuned against the confusion matrix and business risk appetite in `src/risk_score.py`.
- `scored_customers` is also written back into the database so the dashboard and SQL layer can query it directly.
- This is the final notebook in the pipeline -- proceed to `dashboard/app.py` to explore the results interactively.